# Is the admitted feature set better than a random set of the same size?

*Setup cells below are carried from the shared analysis so this notebook runs on its own.*


In [1]:
import os
import statistics as st

import numpy as np
import polars as pl
from google.cloud import bigquery
from sklearn.metrics import average_precision_score, roc_auc_score

from fraud_detection.core.config import resolve_repo_path
from fraud_detection.core.feature_contract import FeatureContract
from fraud_detection.core.feature_contract.admission import load_admission_rules
from fraud_detection.core.schema import (
    CLIENT_ENTITY_ANCHOR,
    CLIENT_ENTITY_COMPONENTS,
    MODEL_INPUT_TABLE,
    SPLIT_TABLE,
)
from fraud_detection.evaluation.entity_purity import Anchor, EntityKey, seen_entity_flag
from fraud_detection.feature_engineering.derivations import apply_derivations
from fraud_detection.training.data import SplitFrame, load_raw_split, prepare_features
from fraud_detection.training.model import train_lightgbm

PROJECT = os.environ["GCP_PROJECT_ID"]
SEEDS = [42, 7, 1337, 2024, 91]
NOISE_SD = {"roc_auc": 0.0029, "pr_auc": 0.0065}
RESOLUTION = {m: sd * (2 / len(SEEDS)) ** 0.5 for m, sd in NOISE_SD.items()}
print(f"this design resolves ROC-AUC ±{RESOLUTION['roc_auc']:.4f}, PR-AUC ±{RESOLUTION['pr_auc']:.4f}")

this design resolves ROC-AUC ±0.0018, PR-AUC ±0.0041


In [2]:
bq = bigquery.Client(project=PROJECT)
contract = FeatureContract.from_json(resolve_repo_path("references/feature-contract.json").read_text())
rules = load_admission_rules()
tables = {"model_input_table": MODEL_INPUT_TABLE, "split_table": SPLIT_TABLE}
raw = {s: load_raw_split(bq, PROJECT, s, **tables) for s in ("train", "val", "test")}
derived = {s: apply_derivations(f, rules.derivations) for s, f in raw.items()}

key = EntityKey(columns=CLIENT_ENTITY_COMPONENTS, anchors=(Anchor(CLIENT_ENTITY_ANCHOR),))
seen = {s: seen_entity_flag(raw["train"], f, key).fill_null(False).cast(pl.Boolean)
        for s, f in raw.items()}
prepared = {s: prepare_features(f) for s, f in derived.items()}

admitted = [c for c in contract.training_features() if c in prepared["train"].columns]
rejected = [c.name for c in contract.columns
            if not c.admitted and c.name in prepared["train"].columns]
print(f"admitted {len(admitted)}, rejected and available {len(rejected)}")

admitted 205, rejected and available 297


In [3]:
WINNING = {"num_leaves": [96], "learning_rate": [0.05], "feature_fraction": [0.6],
           "bagging_fraction": [0.7], "min_child_samples": [80]}


def split_on(columns, name):
    frame = derived[name]
    return SplitFrame(
        features=prepared[name].select(columns),
        labels=frame.get_column("isFraud").cast(pl.Int8),
        amounts=frame.get_column("TransactionAmt").cast(pl.Float64),
        seen_in_train=seen[name],
    )


def evaluate(columns, label, seeds=SEEDS):
    """Seed-averaged raw ranking metrics for one column set.

    Raw scores, because calibration is a separate decision and the submission carries the
    raw ones. Seed-averaged, because a single fit cannot resolve anything this notebook
    is asking about.
    """
    splits = {s: split_on(columns, s) for s in ("train", "val", "test")}
    y = splits["test"].labels.to_numpy()
    rows = []
    for seed in seeds:
        m = train_lightgbm(splits["train"], splits["val"], splits["test"],
                           search_space=WINNING, n_iter=1, seed=seed)
        rows.append({"roc_auc": roc_auc_score(y, m.test_scores),
                     "pr_auc": average_precision_score(y, m.test_scores)})
    return {
        "variant": label, "features": len(columns),
        "roc_mean": st.mean(r["roc_auc"] for r in rows),
        "roc_sd": st.stdev(r["roc_auc"] for r in rows),
        "pr_mean": st.mean(r["pr_auc"] for r in rows),
        "pr_sd": st.stdev(r["pr_auc"] for r in rows),
    }

### Is the admitted set actually the better half?

Three column sets of **the same size**, so the comparison is about *which* columns and not
*how many*:

- **admitted** — what the contract chose;
- **rejected sample** — the same number of columns, drawn from what it threw away;
- **random sample** — the same number, drawn from everything available.

If the audits are selecting on signal, admitted > random > rejected. If admitted ≈ random,
the audits are choosing no better than chance, and that would be the most important
sentence in this repository.

In [4]:
rng = np.random.default_rng(0)
available = sorted(set(admitted) | set(rejected))
n = len(admitted)

variants = {
    "admitted (the contract)": admitted,
    "rejected, same size": list(rng.choice(rejected, size=min(n, len(rejected)), replace=False)),
    "random, same size": list(rng.choice(available, size=n, replace=False)),
    "everything available": available,
}
results = pl.DataFrame([evaluate(cols, name) for name, cols in variants.items()])
base = results.filter(pl.col("variant") == "admitted (the contract)").row(0, named=True)
results.with_columns(
    roc_delta=(pl.col("roc_mean") - base["roc_mean"]).round(4),
    resolvable=((pl.col("roc_mean") - base["roc_mean"]).abs() > RESOLUTION["roc_auc"]),
).sort("roc_mean", descending=True)

variant,features,roc_mean,roc_sd,pr_mean,pr_sd,roc_delta,resolvable
str,i64,f64,f64,f64,f64,f64,bool
"""everything available""",502,0.901301,0.003418,0.548939,0.012069,0.0062,true
"""random, same size""",205,0.897485,0.003601,0.535978,0.006963,0.0024,true
"""admitted (the contract)""",205,0.895133,0.002147,0.516071,0.006195,0.0,false
"""rejected, same size""",205,0.861709,0.003015,0.459847,0.002357,-0.0334,true
